In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load & Base Clean
df_sales = pd.read_csv("sales.csv")
df_sales["date"] = pd.to_datetime(df_sales["date"])
df_sales["year"] = df_sales["date"].dt.year
df_sales["month"] = df_sales["date"].dt.month
df_sales["day"] = df_sales["date"].dt.day
df_sales["is_state_holiday"] = (
    (df_sales["state_holiday"].astype(str).str.strip() != "0")
).astype(int)

df_trainable = df_sales[(df_sales["open"] == 1) & (df_sales["sales"] > 0)]

# 2. Train/Test Split FIRST (To avoid data leakage during target encoding)
X = df_trainable.drop(columns=["sales", "date", "state_holiday", "open"])
y = df_trainable["sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Target Encoding for store_ID (Capture individual store behavior safely)
# Calculate mean sales per store using the training set
store_means = y_train.groupby(X_train["store_ID"]).mean()

# Map the mean sales back to both train and test sets
X_train["store_avg_sales"] = X_train["store_ID"].map(store_means)
X_test["store_avg_sales"] = X_test["store_ID"].map(store_means)



In [4]:
# Fill any missing values in test set (in case a store ID only appeared in test)
global_mean = y_train.mean()
X_test["store_avg_sales"] = X_test["store_avg_sales"].fillna(global_mean)

# Now we can safely drop the raw 'store_ID' column
X_train = X_train.drop(columns=["store_ID"])
X_test = X_test.drop(columns=["store_ID"])

# 4. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Evaluate Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=15, random_state=42, n_jobs=-1
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42
    )
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = r2_score(y_test, y_pred)

print("\n=== Model Comparison with Store Info (R² Scores) ===")
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {score:.4f}")


=== Model Comparison with Store Info (R² Scores) ===
Random Forest: 0.9292
XGBoost: 0.9215
Linear Regression: 0.8181
